In [ ]:
# ABOUTME: Interactive notebook demonstrating cross-modal embeddings with text and images
# ABOUTME: Uses SigLIP2 via open_clip to show zero-shot classification and text-image search

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torchvision
import open_clip
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity
from PIL import Image
from ipywidgets import interact, widgets
import jscatter
import base64
from io import BytesIO

%matplotlib widget


def pil_to_data_uri(img, size=128, fmt='JPEG'):
    """Convert a PIL image to a base64 data URI for jscatter tooltip preview."""
    thumb = img.copy()
    thumb.thumbnail((size, size))
    buf = BytesIO()
    thumb.save(buf, format=fmt)
    b64 = base64.b64encode(buf.getvalue()).decode('utf-8')
    mime = 'image/jpeg' if fmt == 'JPEG' else 'image/png'
    return f"data:{mime};base64,{b64}"

# Where Words Meet Pictures: Cross-Modal Embeddings

**What if words and pictures spoke the same language?**

In the previous notebooks, we embedded images into a vector space. But what if we could also embed *text* into the **same** space — so that a photo of a cat and the sentence "a photo of a cat" land near each other?

This is exactly what models like **CLIP** and **SigLIP** do. They have two encoders:
- An **image encoder** that converts pixels → vectors
- A **text encoder** that converts words → vectors

Both produce vectors in the **same embedding space**. This means we can:
1. **Search images with text**: type "sunset over mountains" and find matching photos
2. **Classify without training**: describe the categories in words, and the model figures out which images match
3. **Compare anything to anything**: images to text, text to text, images to images

## How Does It Work?

The model is trained on **billions of text-image pairs** scraped from the web (e.g., image + caption from a webpage). The training objective:

- **Matching pairs** (e.g., a cat photo + "a cat sitting on a windowsill") should have **high** cosine similarity
- **Non-matching pairs** (e.g., a cat photo + "a red sports car") should have **low** cosine similarity

After training, the model has learned a shared language: the vector for "dog" points in the same direction as the vector for a photo of a dog.

```
"a photo of a dog" ──→ Text Encoder ──→ [0.12, -0.34, 0.56, ...]
                                              ↑ close!
🐕 (photo)         ──→ Image Encoder ──→ [0.11, -0.31, 0.58, ...]
```

In [ ]:
# ============================================================================
# CONFIGURATION
# ============================================================================

# SigLIP2 model — a modern, efficient vision-language model
MODEL_NAME = 'ViT-B-16-SigLIP2-256'
PRETRAINED = 'webli'

N_IMAGES = 30           # Total pet images (half dogs, half cats)
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DATA_DIR = "./data"
THUMBNAIL_SIZE = 128

# Text prompts for zero-shot classification and visualization
TEXT_PROMPTS = [
    "a photo of a dog",
    "a photo of a cat",
    "a cute puppy playing",
    "a fluffy kitten",
    "a large dog outdoors",
    "a small cat sleeping",
]

print(f"Device: {DEVICE}")
print(f"Model: {MODEL_NAME}")
print(f"Text prompts: {len(TEXT_PROMPTS)}")

In [ ]:
# ============================================================================
# LOAD SIGLIP2 MODEL
# ============================================================================

print(f"Loading {MODEL_NAME} (this may download ~300MB on first run)...")

model, _, preprocess = open_clip.create_model_and_transforms(
    MODEL_NAME, pretrained=PRETRAINED, device=DEVICE
)
tokenizer = open_clip.get_tokenizer(MODEL_NAME)
model.eval()

print(f"Model loaded on {DEVICE}")

In [ ]:
# ============================================================================
# LOAD IMAGES AND COMPUTE EMBEDDINGS
# ============================================================================

np.random.seed(SEED)
torch.manual_seed(SEED)

# Load Oxford Pets
dataset = torchvision.datasets.OxfordIIITPet(
    root=DATA_DIR, split='trainval',
    target_types='binary-category',
    download=True
)

cat_indices = [i for i, (_, label) in enumerate(dataset) if label == 0]
dog_indices = [i for i, (_, label) in enumerate(dataset) if label == 1]

n_per_class = N_IMAGES // 2
sampled_indices = np.concatenate([
    np.random.choice(cat_indices, n_per_class, replace=False),
    np.random.choice(dog_indices, n_per_class, replace=False),
])
np.random.shuffle(sampled_indices)

images = []
image_labels = []
label_names = {0: "Cat", 1: "Dog"}

for idx in sampled_indices:
    img, label = dataset[idx]
    images.append(img)
    image_labels.append(label_names[label])

# Compute image embeddings
print("Computing image embeddings...")
image_tensors = torch.stack([preprocess(img) for img in images]).to(DEVICE)
with torch.no_grad():
    image_embeddings = model.encode_image(image_tensors)
    image_embeddings = image_embeddings / image_embeddings.norm(dim=-1, keepdim=True)
image_embeddings_np = image_embeddings.cpu().numpy()

# Compute text embeddings
print("Computing text embeddings...")
text_tokens = tokenizer(TEXT_PROMPTS).to(DEVICE)
with torch.no_grad():
    text_embeddings = model.encode_text(text_tokens)
    text_embeddings = text_embeddings / text_embeddings.norm(dim=-1, keepdim=True)
text_embeddings_np = text_embeddings.cpu().numpy()

print(f"Image embeddings: {image_embeddings_np.shape}")
print(f"Text embeddings:  {text_embeddings_np.shape}")
print(f"Same dimension! Both live in a {image_embeddings_np.shape[1]}-dim space.")

## Text and Images in the Same Space

Let's project all embeddings (text + images) to 2D with PCA and see if matching concepts land near each other. Each point is either an image or a text prompt.

In [ ]:
# ============================================================================
# COMBINED VISUALIZATION: TEXT + IMAGES
# ============================================================================

# Combine all embeddings
combined_embeddings = np.vstack([image_embeddings_np, text_embeddings_np])

# PCA on combined space
pca = PCA(n_components=2)
combined_2d = pca.fit_transform(combined_embeddings)

n_imgs = len(images)
n_texts = len(TEXT_PROMPTS)

# Generate data URIs for image thumbnails (empty string for text points)
thumbnail_uris = [pil_to_data_uri(img, size=THUMBNAIL_SIZE) for img in images]
all_thumbnail_uris = thumbnail_uris + [''] * n_texts

# Build DataFrame
modalities = ['image'] * n_imgs + ['text'] * n_texts
point_labels = image_labels + ['text'] * n_texts
descriptions = [f"Image: {label}" for label in image_labels] + TEXT_PROMPTS

df = pd.DataFrame({
    'pca_x': combined_2d[:, 0],
    'pca_y': combined_2d[:, 1],
    'modality': pd.Categorical(modalities),
    'animal': pd.Categorical(point_labels),
    'description': descriptions,
    'thumbnail': all_thumbnail_uris,
})

scatter_combined = jscatter.Scatter(
    data=df, x='pca_x', y='pca_y',
    height=500,
)
scatter_combined.color(by='modality', map={'image': '#3498db', 'text': '#e74c3c'})
scatter_combined.size(8)
scatter_combined.tooltip(
    enable=True,
    properties=['modality', 'description'],
    preview='thumbnail',
    preview_type='image',
    preview_image_size='contain',
)
scatter_combined.legend(True)
scatter_combined.show()

Notice how the text prompt "a photo of a dog" lands right in the middle of the dog image cluster, and "a photo of a cat" lands among the cats. **Words and pictures are neighbors in embedding space.**

## Text-Image Similarity Matrix

Let's see how similar each text prompt is to each image. This matrix is the foundation of cross-modal search.

In [ ]:
# ============================================================================
# TEXT-IMAGE SIMILARITY MATRIX
# ============================================================================

# Compute cross-modal similarity: text_prompts x images
cross_sim = cosine_similarity(text_embeddings_np, image_embeddings_np)

fig, ax = plt.subplots(figsize=(14, 5))
sns.heatmap(
    cross_sim,
    xticklabels=[f"{i} ({l})" for i, l in enumerate(image_labels)],
    yticklabels=TEXT_PROMPTS,
    annot=True,
    fmt='.2f',
    cmap='RdYlBu_r',
    center=0.0,
    cbar_kws={'label': 'Cosine Similarity'},
    ax=ax,
)
ax.set_title('Text → Image Similarity', fontsize=14, fontweight='bold')
ax.set_xlabel('Images')
ax.set_ylabel('Text Prompts')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.tight_layout()
plt.show()

## Zero-Shot Classification: Classifying Without Training

Here's the magic: we can classify images **without any labeled training data**. Just describe the categories in words, compute similarities, and pick the closest match.

Pick an image below and see which text prompt matches it best.

In [ ]:
# ============================================================================
# INTERACTIVE ZERO-SHOT CLASSIFICATION
# ============================================================================

fig_zs, (ax_img, ax_bar) = plt.subplots(1, 2, figsize=(12, 5),
                                          gridspec_kw={'width_ratios': [1, 2]})

query_options = {f"#{i} ({image_labels[i]})": i for i in range(len(images))}

@interact(
    query=widgets.Dropdown(
        options=query_options,
        value=0,
        description='Image:',
        style={'description_width': 'initial'},
    ),
)
def zero_shot_classify(query):
    ax_img.clear()
    ax_bar.clear()

    # Show the query image
    ax_img.imshow(images[query])
    ax_img.set_title(f'Query: {image_labels[query]}', fontsize=12, fontweight='bold')
    ax_img.axis('off')

    # Compute similarity to all text prompts
    sims = cross_sim[:, query]
    best_idx = np.argmax(sims)

    # Bar chart of similarities
    colors = ['forestgreen' if i == best_idx else 'steelblue' for i in range(len(TEXT_PROMPTS))]
    bars = ax_bar.barh(range(len(TEXT_PROMPTS)), sims, color=colors, height=0.6)
    ax_bar.set_yticks(range(len(TEXT_PROMPTS)))
    ax_bar.set_yticklabels(TEXT_PROMPTS, fontsize=10)
    ax_bar.set_xlabel('Cosine Similarity', fontsize=11)
    ax_bar.set_title('Similarity to Each Text Prompt', fontsize=12, fontweight='bold')

    for bar, val in zip(bars, sims):
        ax_bar.text(val + 0.005, bar.get_y() + bar.get_height() / 2,
                    f'{val:.3f}', va='center', fontsize=10, fontweight='bold')

    ax_bar.set_xlim(0, max(sims) * 1.15)

    fig_zs.suptitle(f'Best match: "{TEXT_PROMPTS[best_idx]}"', fontsize=13,
                     fontweight='bold', color='forestgreen')
    fig_zs.tight_layout()
    fig_zs.canvas.draw_idle()

## Zero-Shot Accuracy: How Well Does It Work?

Let's test the model on all our images. For each image, we'll check: does the most similar prompt between "a photo of a dog" and "a photo of a cat" match the true label?

In [ ]:
# ============================================================================
# ZERO-SHOT ACCURACY
# ============================================================================

# Use just the two basic prompts for binary classification
class_prompts = ["a photo of a dog", "a photo of a cat"]
class_tokens = tokenizer(class_prompts).to(DEVICE)

with torch.no_grad():
    class_text_emb = model.encode_text(class_tokens)
    class_text_emb = class_text_emb / class_text_emb.norm(dim=-1, keepdim=True)

class_text_np = class_text_emb.cpu().numpy()

# Classify each image
class_sim = cosine_similarity(class_text_np, image_embeddings_np)  # [2, N_images]
predictions = ["Dog" if class_sim[0, i] > class_sim[1, i] else "Cat"
               for i in range(len(images))]

correct = sum(p == t for p, t in zip(predictions, image_labels))
accuracy = correct / len(images)

print(f"Zero-shot classification accuracy: {correct}/{len(images)} = {accuracy:.1%}")
print()

# Show misclassifications if any
misclassified = [(i, image_labels[i], predictions[i])
                 for i in range(len(images))
                 if predictions[i] != image_labels[i]]

if misclassified:
    print(f"Misclassified {len(misclassified)} images:")
    for idx, true_label, pred_label in misclassified:
        print(f"  Image #{idx}: true={true_label}, predicted={pred_label}")
else:
    print("Perfect classification — every image was correctly matched to its text prompt!")

## The Power of This Approach

Think about what just happened:
- We **never showed the model any labeled training examples** of dogs or cats
- We just described the categories in plain English: "a photo of a dog", "a photo of a cat"
- The model matched images to text descriptions **purely based on learned semantic similarity**

This is **zero-shot classification** — and it works for *any* categories you can describe in words. Want to classify images as "happy" vs "sad"? Just change the text prompts. No retraining needed.

## Summary: The Embeddings Journey

| Notebook | Key Concept | What We Learned |
|----------|-------------|-----------------|
| **03** | Embeddings | Neural networks convert images to meaningful vectors |
| **04** | Layer hierarchy | Early layers see colors, late layers see concepts |
| **05** | Similarity search | Cosine similarity powers search engines |
| **06** | Cross-modal | Text and images share the same embedding space |

These concepts are the foundation of modern AI:
- **Search engines** (Google Images, Spotify, Netflix) use embedding similarity
- **RAG** (Retrieval-Augmented Generation) embeds documents to find relevant context for LLMs
- **Recommendation systems** find similar items in embedding space
- **Zero-shot classification** lets you classify without training data